# A3-M1 · 02 Hugging Face 토크나이저 직접 만들기
`tokenizers`로 WordPiece·바이트 BPE 토크나이저를 학습합니다. 먼저 `python make_corpus.py`로 `shop_corpus.txt`를 만드세요.

## 04-2. 준비 — 말뭉치 읽기

In [2]:
from pathlib import Path
lines = Path("shop_corpus.txt").read_text(encoding="utf-8").splitlines()
print(len(lines), "줄")
print(lines[:3])

2000 줄
['냉장고 교환이 가능한가요?', '무선이어폰 색상을 바꾸고 싶어요', '에어컨 배송지 변경하고 싶어요']


## 04-3. WordPiece 토크나이저 학습

In [3]:
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers, processors, decoders

SPECIAL = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
wp = Tokenizer(models.WordPiece(unk_token="[UNK]"))                 # ① 모델: WordPiece
wp.normalizer = normalizers.Sequence([normalizers.NFKC(), normalizers.Lowercase()])   # ② 정규화
wp.pre_tokenizer = pre_tokenizers.BertPreTokenizer()               # ③ 미리 나누기: 공백·문장부호
trainer = trainers.WordPieceTrainer(vocab_size=800, min_frequency=2, special_tokens=SPECIAL)
wp.train_from_iterator(lines, trainer=trainer)                     # ④ 학습
print("어휘 수:", wp.get_vocab_size())
print("특수 토큰 번호:", {t: wp.token_to_id(t) for t in SPECIAL})

어휘 수: 632
특수 토큰 번호: {'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3, '[MASK]': 4}


## 05-1. encode 결과(Encoding) 읽기

In [4]:
e = wp.encode("냉장고 배송비는 얼마인가요? FREE shipping")
print("tokens :", e.tokens)
print("ids    :", e.ids)
print("offsets:", e.offsets)
print("mask   :", e.attention_mask)
print("원문 조각:", [ "냉장고 배송비는 얼마인가요? FREE shipping"[s:t] for s, t in e.offsets])
print("모르는 글자:", wp.encode("냉장고 ♨ 배송").tokens)

tokens : ['냉장고', '배송비는', '얼마인가요', '?', 'free', 'shipping']
ids    : [449, 447, 448, 17, 496, 500]
offsets: [(0, 3), (4, 8), (9, 14), (14, 15), (16, 20), (21, 29)]
mask   : [1, 1, 1, 1, 1, 1]
원문 조각: ['냉장고', '배송비는', '얼마인가요', '?', 'FREE', 'shipping']
모르는 글자: ['냉장고', '[UNK]', '배송']


## 05-2. 후처리 — [CLS]·[SEP] 붙이기와 문장 쌍

In [5]:
cls_id, sep_id = wp.token_to_id("[CLS]"), wp.token_to_id("[SEP]")
wp.post_processor = processors.TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", cls_id), ("[SEP]", sep_id)])
wp.decoder = decoders.WordPiece()

one = wp.encode("배송비는 얼마인가요?")
print("한 문장 :", one.tokens)
pair = wp.encode("배송비는 얼마인가요?", "3만 원 이상 무료 배송입니다.")
print("문장 쌍 :", pair.tokens)
print("type_ids:", pair.type_ids)
print("decode  :", wp.decode(pair.ids))

한 문장 : ['[CLS]', '배송비는', '얼마인가요', '?', '[SEP]']
문장 쌍 : ['[CLS]', '배송비는', '얼마인가요', '?', '[SEP]', '3만', '원', '이상', '무료', '배송입니다', '.', '[SEP]']
type_ids: [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]
decode  : 배송비는 얼마인가요? 3만 원 이상 무료 배송입니다.


## 05-3. padding과 truncation — 여러 문장을 한 번에

In [6]:
wp.enable_padding(pad_id=wp.token_to_id("[PAD]"), pad_token="[PAD]")
wp.enable_truncation(max_length=10)
batch = wp.encode_batch(["배송 언제 와요?", "냉장고 반품하려면 어떻게 하나요? 설치 기사님 방문 일정은요?"])
for b in batch:
    print(b.tokens)
    print("  mask:", b.attention_mask, "| 잘린 나머지:", [o.tokens for o in b.overflowing][:1])
wp.no_padding(); wp.no_truncation()

['[CLS]', '배송', '언제', '[UNK]', '?', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
  mask: [1, 1, 1, 1, 1, 1, 0, 0, 0, 0] | 잘린 나머지: []
['[CLS]', '냉장고', '반품하려면', '어떻게', '하나요', '?', '설치', '기사님', '방문', '[SEP]']
  mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] | 잘린 나머지: [['[CLS]', '일정은요', '?', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']]


## 06-1. BPE(바이트 단위) 토크나이저 학습 — GPT 방식과 비교

In [7]:
bpe = Tokenizer(models.BPE())
bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe.decoder = decoders.ByteLevel()
bpe.train_from_iterator(lines, trainers.BpeTrainer(vocab_size=800, min_frequency=2,
                        special_tokens=["<pad>", "<eos>"], initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
s = "냉장고 배송비는 얼마인가요? ♨"
print("WordPiece:", wp.encode(s).tokens)
print("ByteBPE  :", bpe.encode(s).tokens)
print("ByteBPE 복원:", bpe.decode(bpe.encode(s).ids))

WordPiece: ['[CLS]', '냉장고', '배송비는', '얼마인가요', '?', '[UNK]', '[SEP]']
ByteBPE  : ['ëĥīìŀ¥ê³ł', 'Ġë°°ìĨ¡ë¹ĦëĬĶ', 'Ġìĸ¼ë§ĪìĿ¸ê°ĢìļĶ', '?', 'Ġ', 'â', 'Ļ', '¨']
ByteBPE 복원: 냉장고 배송비는 얼마인가요? ♨


## 06-2. 저장과 불러오기 — tokenizer.json 한 파일

In [8]:
wp.save("shop_wordpiece.json")
again = Tokenizer.from_file("shop_wordpiece.json")
print("파일 크기:", Path("shop_wordpiece.json").stat().st_size, "바이트")
print("같은 결과?", again.encode("배송 언제 와요?").ids == wp.encode("배송 언제 와요?").ids)

파일 크기: 15599 바이트
같은 결과? True
